# Demo: Wilcoxon Signed-Rank Test for Two Models

This notebook simulates evaluation scores for two hypothetical models on the same test set and demonstrates the Wilcoxon signed-rank test step by step.

You can control:
1. test set size `n`
2. score range (for example 1-10)
3. average advantage of model A over model B


In [ ]:
import random

# --- User settings ---
n = 40
score_min = 1
score_max = 10
mean_advantage_a = 0.1
noise_std = 1.2
seed = 7


## 1) Simulate paired scores

For each test case, we sample a score for model B, then generate model A by adding random noise centered on the chosen average advantage. Scores are rounded and clipped to the allowed discrete range.


In [ ]:
rng = random.Random(seed)

def clip_score(x, min_score, max_score):
    return max(min_score, min(max_score, x))

scores_a = []
scores_b = []
for _ in range(n):
    b = rng.randint(score_min, score_max)
    raw_a = round(b + rng.gauss(mean_advantage_a, noise_std))
    a = clip_score(raw_a, score_min, score_max)
    scores_b.append(b)
    scores_a.append(a)

pairs = list(zip(scores_a, scores_b))
mean_diff = sum(a - b for a, b in pairs) / n

print(f'Generated {n} paired scores.')
print(f'Observed average (A - B): {mean_diff:.3f}')
print('First 10 pairs (A, B):', pairs[:10])


## 2) Compute paired differences and remove zeros

For each pair, compute `d_i = score_A_i - score_B_i`. Differences equal to zero are ignored by the Wilcoxon signed-rank test.


In [ ]:
diffs = [a - b for a, b in pairs]
nonzero_diffs = [d for d in diffs if d != 0]

print(f'Total pairs: {len(diffs)}')
print(f'Non-zero differences used by Wilcoxon: {len(nonzero_diffs)}')
print('First 20 differences:', diffs[:20])


## 3) Visualize paired scores and differences

These plots help build intuition before looking at the rank-based test statistic:
- **Left:** score distribution for model A vs model B
- **Right:** distribution of paired differences `(A - B)`


In [ ]:
from collections import Counter

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

if plt is None:
    print('Matplotlib not installed in this environment; install from requirements.txt to see plots.')
else:
    score_values = list(range(score_min, score_max + 1))
    counts_a = Counter(scores_a)
    counts_b = Counter(scores_b)
    heights_a = [counts_a.get(v, 0) for v in score_values]
    heights_b = [counts_b.get(v, 0) for v in score_values]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Plot 1: Score distributions for A and B
    bar_width = 0.4
    x_left = [v - bar_width / 2 for v in score_values]
    x_right = [v + bar_width / 2 for v in score_values]
    axes[0].bar(x_left, heights_a, width=bar_width, label="Model A", alpha=0.8)
    axes[0].bar(x_right, heights_b, width=bar_width, label="Model B", alpha=0.8)
    axes[0].set_title('Score distribution by model')
    axes[0].set_xlabel('Score')
    axes[0].set_ylabel('Count')
    axes[0].set_xticks(score_values)
    axes[0].legend()

    # Plot 2: Differences histogram
    min_d = min(diffs)
    max_d = max(diffs)
    bins = [x - 0.5 for x in range(min_d, max_d + 2)]
    axes[1].hist(diffs, bins=bins, edgecolor="black", alpha=0.8)
    axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
    axes[1].set_title('Distribution of paired differences (A - B)')
    axes[1].set_xlabel('Difference')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()


## 4) Rank `|d_i|` (average ranks for ties), restore signs, and compute `W+`, `W-`


In [ ]:
def average_ranks(values):
    """Return ranks (1..n) with average ranks for tied values."""
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    next_rank = 1
    while i < len(order):
        j = i
        v = values[order[i]]
        while j < len(order) and values[order[j]] == v:
            j += 1
        count = j - i
        avg_rank = (next_rank + (next_rank + count - 1)) / 2
        for k in range(i, j):
            ranks[order[k]] = avg_rank
        next_rank += count
        i = j
    return ranks

if nonzero_diffs:
    abs_diffs = [abs(d) for d in nonzero_diffs]
    ranks = average_ranks(abs_diffs)
    signed_ranks = [r if d > 0 else -r for d, r in zip(nonzero_diffs, ranks)]
    W_plus = sum(r for r in signed_ranks if r > 0)
    W_minus = -sum(r for r in signed_ranks if r < 0)
    T_stat = min(W_plus, W_minus)

    print(f"{'i':>2} {'d_i':>4} {'|d_i|':>6} {'rank':>7} {'signed rank':>12}")
    for i, (d, ad, r, sr) in enumerate(zip(nonzero_diffs, abs_diffs, ranks, signed_ranks), start=1):
        if i > 12:
            print('...')
            break
        print(f"{i:>2} {d:+4d} {ad:>6} {r:>7.1f} {sr:>12.1f}")

    print(f'W+ = {W_plus:.3f}')
    print(f'W- = {W_minus:.3f}')
    print(f'T = min(W+, W-) = {T_stat:.3f}')
else:
    ranks = []
    W_plus = 0.0
    W_minus = 0.0
    T_stat = 0.0
    print('All differences are zero, so the test has no informative ranks.')


## 5) Compute p-value from the null distribution

Under the null hypothesis (median difference = 0), each non-zero difference sign is equally likely to be positive or negative.

Below we compute an **exact two-sided p-value** by dynamic programming over all possible sign assignments of the observed ranks.


In [ ]:
def exact_two_sided_p_value_from_ranks(ranks, observed_T):
    """Exact two-sided p-value for T=min(W+,W-) given fixed ranks."""
    if not ranks:
        return 1.0

    # Ranks are integer/half-integer when using average ties, so scale by 2.
    scaled_ranks = [int(round(r * 2)) for r in ranks]
    total = sum(scaled_ranks)
    observed_scaled_T = int(round(observed_T * 2))

    # dist[s] = number of sign assignments with W+ == s
    dist = {0: 1}
    for rv in scaled_ranks:
        nxt = {}
        for s, count in dist.items():
            nxt[s] = nxt.get(s, 0) + count
            s_plus = s + rv
            nxt[s_plus] = nxt.get(s_plus, 0) + count
        dist = nxt

    extreme = sum(
        count
        for s, count in dist.items()
        if min(s, total - s) <= observed_scaled_T
    )
    return extreme / (2 ** len(scaled_ranks))

p_value_two_sided = exact_two_sided_p_value_from_ranks(ranks, T_stat)
print(f'Exact two-sided p-value: {p_value_two_sided:.6f}')
print('Interpretation: smaller p-values provide stronger evidence that the models differ.')


## 6) Optional cross-check with SciPy (if installed)


In [ ]:
try:
    from scipy.stats import wilcoxon

    if nonzero_diffs:
        scipy_result = wilcoxon(nonzero_diffs, zero_method='wilcox', alternative='two-sided', method='auto')
        print('SciPy statistic:', scipy_result.statistic)
        print('SciPy p-value  :', scipy_result.pvalue)
    else:
        print('SciPy check skipped: all differences are zero.')
except ModuleNotFoundError:
    print('SciPy not installed in this environment; optional check skipped.')


### Notes
- Change `mean_advantage_a` and rerun to see how evidence changes.
- Increase `n` to see how larger test sets can detect smaller effects.
- This demo uses discrete scores and paired comparisons, matching common model-eval setups.
